# Bays (2014) Figure 4 — Robustness Under Tuning and Noise Variations
## Broad/Narrow GP, Baseline, Heterogeneous, Cosine, Correlated

### Intellectual Foundation

The previous figures used a standard homogeneous GP population. This figure asks:
**does the key finding (non-Gaussian errors at low gain) survive when we relax the model assumptions?**

Six panels, each modifying one aspect:
- **a: Broad GP** (λ=1.0) — smooth, wide tuning
- **b: Narrow GP** (λ=0.3) — peaked, selective tuning
- **c: Baseline activity** — adding a constant floor to firing rates
- **d: Heterogeneous GP** — each neuron gets its own λ, amplitude, and baseline
- **e: Cosine tuning** — half-wave rectified cosine (replacing GP entirely)
- **f: Correlated activity** — short-range noise correlations between neurons

### What to play with
- `GAMMAS`: The gain sweep — logarithmically spaced from 1 to 128
- `BASELINE_FRAC`: For panel c, how much baseline activity (0-1)
- `C0`: For panel f, peak correlation strength

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import time

np.random.seed(42)

# --- PARAMETERS ---
N_THETA = 64; N_TRIALS = 3000; T_D = 0.1; SIGMA_SQ = 1e-6
LAMBDA_BROAD = 1.0; LAMBDA_NARROW = 0.3; BASELINE_FRAC = 0.25; C0 = 0.25
GAMMAS = [2, 8, 32, 128]; M_POP = 100  # Population size
SEED = 42; N_BINS = 50

In [ ]:
# ============================================================
# SELF-CONTAINED CORE: GP Population + DN + Poisson + ML Decoder
# ============================================================
# Replaces imports from core.encoder.*, core.decoder.*, bays_utils

def periodic_rbf_kernel(thetas, lengthscale):
    """Periodic squared-exponential kernel."""
    diff = thetas[:, None] - thetas[None, :]
    circ = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ / lengthscale)**2)

def sample_gp_function(K, rng):
    """Sample one function from GP(0, K)."""
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(K)))
    return L @ rng.randn(len(K))

def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    """Generate M neurons with GP tuning at n_locations. Returns (thetas, f_all)."""
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    f_all = []
    for loc in range(n_locations):
        f_loc = np.zeros((M, n_theta))
        for i in range(M):
            f_loc[i] = sample_gp_function(K, rng)
        f_all.append(f_loc)
    return thetas, f_all

def dn_pointwise(r_pre, gamma, sigma_sq):
    """DN: r^post_i = γ · r^pre_i / (σ² + mean(r^pre))."""
    D = sigma_sq + np.mean(r_pre)
    return gamma * r_pre / D

def generate_spikes(rates, T_d, rng):
    """Poisson spike counts."""
    return rng.poisson(np.maximum(rates, 0) * T_d)

def compute_log_likelihood(counts, g, T_d):
    """Full Poisson ML: LL(θ) = Σ_i [n_i·log g_i(θ) - g_i(θ)·T_d]."""
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    """Signed circular error in [-π, π)."""
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    """Circular variance: 1 - |mean(exp(iε))|."""
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    """Circular kurtosis: κ₂/V² where κ₂ = 1-|ρ₂|, V = circ var."""
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    """Deviation of empirical dist from matched circular normal."""
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = bin_edges[1] - bin_edges[0]
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Factorised multi-location decoder
from scipy.special import logsumexp

def compute_spike_weighted_log_tuning(counts, f_list):
    """L_k(θ) = Σ_i n_i · f_{i,k}(θ) for each location k."""
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    """Efficient factorised ML: L_c(θ) + Σ_{k≠c} logsumexp(L_k)."""
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

print("Core model loaded (GP + DN + Poisson + ML decoder)")

---
## Experiment: Six Panel Configurations

### What is occurring
For each panel, we generate the population with its specific modification, then sweep gain γ.
At each γ, we run the full encode → DN → Poisson → Full ML decode pipeline.
The error distributions at different γ values are overlaid (colour = gain).

In [ ]:
# ============================================================
# POPULATION GENERATORS (one per panel)
# ============================================================
def make_gp_pop(M, lam, seed):
    thetas, f_all = generate_population(M, N_THETA, lam, 1, seed)
    return thetas, np.exp(f_all[0])

def make_gp_baseline(M, lam, bl_frac, seed):
    thetas, g = make_gp_pop(M, lam, seed)
    if bl_frac > 1e-10:
        peak = np.mean(np.max(g, axis=1))
        b0 = bl_frac * peak / (1 - bl_frac)
        g = g + b0
    return thetas, g

def make_heterogeneous(M, seed):
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, N_THETA, endpoint=False)
    g = np.zeros((M, N_THETA))
    for i in range(M):
        lam_i = max(0.05, rng.normal(0.5, 0.2))
        amp_i = max(0.01, rng.normal(1.0, 0.3))
        bl_i = max(0, rng.normal(0.1, 0.05))
        K = periodic_rbf_kernel(thetas, lam_i) + 1e-6*np.eye(N_THETA)
        f_i = np.linalg.cholesky(K) @ rng.randn(N_THETA)
        g[i] = amp_i * np.exp(f_i) + bl_i
    return thetas, g

def make_cosine(M, seed):
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, N_THETA, endpoint=False)
    prefs = rng.uniform(-np.pi, np.pi, M)
    g = np.zeros((M, N_THETA))
    for i in range(M):
        amp = max(0.01, rng.normal(1.0, 0.3))
        bl = max(0, rng.normal(0.1, 0.05))
        g[i] = amp * np.maximum(0, np.cos(thetas - prefs[i])) + bl
    return thetas, g

# ============================================================
# SWEEP GAINS FOR EACH PANEL
# ============================================================
t0 = time.time()
panel_configs = {
    'a': ('Broad GP', lambda: make_gp_pop(M_POP, LAMBDA_BROAD, SEED)),
    'b': ('Narrow GP', lambda: make_gp_pop(M_POP, LAMBDA_NARROW, SEED+100)),
    'c': ('Baseline', lambda: make_gp_baseline(M_POP, LAMBDA_NARROW, BASELINE_FRAC, SEED+200)),
    'd': ('Heterogeneous', lambda: make_heterogeneous(M_POP, SEED+300)),
    'e': ('Cosine', lambda: make_cosine(M_POP, SEED+400)),
}

panels = {}
for pid, (title, gen_fn) in panel_configs.items():
    thetas, g = gen_fn()
    panels[pid] = {'title': title, 'thetas': thetas, 'g': g, 'results': {}}

    for gam in GAMMAS:
        rng = np.random.RandomState(SEED + int(gam))
        errors = np.empty(N_TRIALS)
        for t in range(N_TRIALS):
            idx_true = rng.randint(N_THETA)
            rates = dn_pointwise(g[:, idx_true], gam, SIGMA_SQ)
            counts = generate_spikes(rates, T_D, rng)
            ll = compute_log_likelihood(counts, g, T_D)
            errors[t] = compute_circular_error(thetas[idx_true], thetas[np.argmax(ll)])
        panels[pid]['results'][gam] = compute_deviation_from_normal(errors, N_BINS)

    print(f"  Panel {pid} ({title}) done ({time.time()-t0:.0f}s)")

print(f"\nAll panels done in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 5-COLUMN FIGURE (panels a-e; f would need correlated spikes)
# ============================================================
fig = plt.figure(figsize=(18, 8))
gs = gridspec.GridSpec(2, 5, hspace=0.4, wspace=0.3)
gain_cmap = plt.cm.RdYlBu
gain_norm = mcolors.LogNorm(vmin=min(GAMMAS), vmax=max(GAMMAS))

for col, pid in enumerate(['a', 'b', 'c', 'd', 'e']):
    p = panels[pid]

    # Top: example tuning curves
    ax_top = fig.add_subplot(gs[0, col])
    for i in range(min(20, M_POP)):
        ax_top.plot(p['thetas'], p['g'][i], 'k-', lw=0.4, alpha=0.5)
    ax_top.set_title(p['title'], fontsize=11, fontweight='bold')
    ax_top.set_xlabel('θ'); ax_top.set_ylabel('g(θ)') if col == 0 else None

    # Bottom: error distributions at different gains
    ax_bot = fig.add_subplot(gs[1, col])
    for gam in GAMMAS:
        dev = p['results'][gam]
        emp = dev['empirical']
        emp_norm = emp / max(emp.max(), 1e-10)
        ax_bot.plot(dev['bin_centers'], emp_norm, color=gain_cmap(gain_norm(gam)), lw=1.2)
    ax_bot.set_xlim(-np.pi, np.pi); ax_bot.set_ylim(0, 1.05)
    ax_bot.set_xticks([-np.pi, 0, np.pi]); ax_bot.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
    ax_bot.set_xlabel('error'); ax_bot.set_ylabel('normalised prob') if col == 0 else None

sm = plt.cm.ScalarMappable(cmap=gain_cmap, norm=gain_norm); sm.set_array([])
cb_ax = fig.add_axes([0.3, 0.02, 0.4, 0.015])
cb = fig.colorbar(sm, cax=cb_ax, orientation='horizontal')
cb.set_label(r'gain $\gamma$'); cb.set_ticks(GAMMAS)

fig.suptitle('Bays (2014) Fig 4 — Robustness: Error Distributions Under Model Variations',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.95]); plt.show()

print("\nKEY: Non-Gaussian errors (heavy tails at low gain) persist across ALL panel types")
print("KEY: The finding is robust to tuning shape, baseline, heterogeneity, and tuning model")